# Ch.6 — Evaluation Metrics for Regression

**Goal**: Build a complete evaluation framework for our Ch.5 Ridge model ($38k MAE).

| What | Value |
|------|-------|
| Dataset | California Housing (20,640 districts, 8 features) |
| Best model (Ch.5) | Ridge (degree=2, α=1.0) → $38k MAE |
| This chapter | Validate, diagnose, and understand that $38k |


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)


In [ ]:
# ── Load, split, and fit best model from Ch.5 ──────────────────────────────
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(f"Model: Ridge (degree=2, α=1.0)")
print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")


## Multiple Error Metrics

Each metric reveals a different aspect of model quality.

In [ ]:
# ── Compute all metrics ───────────────────────────────────────────────────
mae = mean_absolute_error(y_test, y_pred) * 100_000
rmse = np.sqrt(mean_squared_error(y_test, y_pred)) * 100_000
r2 = r2_score(y_test, y_pred)
n, p_feat = X_test.shape[0], pipe.named_steps['poly'].n_output_features_
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p_feat - 1)
mape = np.mean(np.abs((y_pred - y_test) / y_test)) * 100

print("═" * 50)
print("  REGRESSION METRICS DASHBOARD")
print("═" * 50)
print(f"  MAE:         ${mae:,.0f}")
print(f"  RMSE:        ${rmse:,.0f}")
print(f"  MAPE:        {mape:.1f}%")
print(f"  R²:          {r2:.4f}")
print(f"  Adjusted R²: {adj_r2:.4f}")
print(f"  RMSE/MAE:    {rmse/mae:.2f}  (1.0=uniform, >1.3=variable)")
print("═" * 50)

## Residual Diagnostics

Aggregate metrics hide structural failures. Residual plots reveal WHERE and HOW the model fails.

In [ ]:
# ── Residual analysis (4 plots) ──────────────────────────────────────────
residuals = (y_test - y_pred) * 100_000

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Predicted vs Actual
axes[0, 0].scatter(y_test * 100_000, y_pred * 100_000, alpha=0.15, s=10, c='steelblue')
lims = [0, 550_000]
axes[0, 0].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[0, 0].set_xlabel('Actual Value ($)', fontsize=11)
axes[0, 0].set_ylabel('Predicted Value ($)', fontsize=11)
axes[0, 0].set_title('Predicted vs Actual\n(Ideal = points on red line)', fontsize=12)
axes[0, 0].legend()

# 2. Residuals vs Predicted
axes[0, 1].scatter(y_pred * 100_000, residuals, alpha=0.15, s=10, c='steelblue')
axes[0, 1].axhline(y=0, color='red', linewidth=2, linestyle='--')
axes[0, 1].set_xlabel('Predicted Value ($)', fontsize=11)
axes[0, 1].set_ylabel('Residual ($)', fontsize=11)
axes[0, 1].set_title('Residuals vs Predicted\n(Look for patterns: curves, fans)', fontsize=12)

# 3. Residual distribution
axes[1, 0].hist(residuals, bins=60, edgecolor='white', color='steelblue', density=True)
x_norm = np.linspace(residuals.min(), residuals.max(), 100)
axes[1, 0].plot(x_norm, stats.norm.pdf(x_norm, residuals.mean(), residuals.std()),
                'r-', linewidth=2, label='Normal fit')
axes[1, 0].axvline(x=0, color='black', linewidth=1, linestyle='--')
axes[1, 0].set_xlabel('Residual ($)', fontsize=11)
axes[1, 0].set_ylabel('Density', fontsize=11)
axes[1, 0].set_title('Residual Distribution\n(Should be symmetric, centered at 0)', fontsize=12)
axes[1, 0].legend()

# 4. Q-Q plot
stats.probplot(residuals, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot\n(Points on line = normally distributed)', fontsize=12)
axes[1, 1].get_lines()[0].set_markersize(3)
axes[1, 1].get_lines()[0].set_alpha(0.4)

plt.tight_layout()
plt.savefig('img/residual_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nResidual statistics:")
print(f"  Mean:   ${residuals.mean():,.0f} (should be ~$0)")
print(f"  Std:    ${residuals.std():,.0f}")
print(f"  Skew:   {stats.skew(residuals):.2f} (0 = symmetric)")
print(f"  Kurt:   {stats.kurtosis(residuals):.2f} (0 = normal tails)")

## Segmented Error Analysis

Average MAE hides disparities across price ranges. Let's break it down.

In [ ]:
# ── MAE by price segment ─────────────────────────────────────────────────
test_df = pd.DataFrame({
    'actual': y_test * 100_000,
    'predicted': y_pred * 100_000,
    'residual': residuals,
    'abs_error': np.abs(residuals)
})

bins = [0, 100_000, 200_000, 300_000, 400_000, 600_000]
labels = ['<$100k', '$100-200k', '$200-300k', '$300-400k', '$400k+']
test_df['segment'] = pd.cut(test_df['actual'], bins=bins, labels=labels)

seg_stats = test_df.groupby('segment', observed=True).agg(
    count=('abs_error', 'count'),
    mae=('abs_error', 'mean'),
    median_error=('abs_error', 'median'),
    mean_residual=('residual', 'mean')
).round(0)

print("MAE by Price Segment:")
print(f"{'Segment':<15} {'Count':>8} {'MAE':>12} {'Median Err':>12} {'Bias':>12}")
print(f"{'-'*15} {'-'*8} {'-'*12} {'-'*12} {'-'*12}")
for seg, row in seg_stats.iterrows():
    bias_dir = '(over)' if row['mean_residual'] < 0 else '(under)'
    print(f"{seg:<15} {int(row['count']):>8} ${row['mae']:>10,.0f} ${row['median_error']:>10,.0f} ${row['mean_residual']:>+10,.0f} {bias_dir}")

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d4edda' if m < 40_000 else '#f8d7da' for m in seg_stats['mae']]
ax.bar(seg_stats.index.astype(str), seg_stats['mae'], color=colors, edgecolor='gray')
ax.axhline(y=40_000, color='green', linestyle='--', linewidth=2, label='$40k target')
ax.set_xlabel('Price Segment', fontsize=12)
ax.set_ylabel('MAE ($)', fontsize=12)
ax.set_title('MAE by Price Segment\n(Model struggles with expensive homes)', fontsize=14)
ax.legend()
for i, (seg, row) in enumerate(seg_stats.iterrows()):
    ax.text(i, row['mae'] + 1000, f"${row['mae']:,.0f}\n(n={int(row['count'])})",
            ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('img/mae_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

## Cross-Validation Stability

Is our $38k MAE stable across different data splits?

In [ ]:
# ── 5-fold cross-validation ───────────────────────────────────────────────
cv_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

cv_mae = -cross_val_score(cv_pipe, X_train, y_train,
                          cv=5, scoring='neg_mean_absolute_error') * 100_000
cv_r2 = cross_val_score(cv_pipe, X_train, y_train, cv=5, scoring='r2')

print("5-Fold Cross-Validation Results:")
print(f"{'Fold':<8} {'MAE':>12} {'R²':>8}")
print(f"{'-'*8} {'-'*12} {'-'*8}")
for i in range(5):
    print(f"Fold {i+1:<3} ${cv_mae[i]:>10,.0f} {cv_r2[i]:>8.4f}")
print(f"{'-'*8} {'-'*12} {'-'*8}")
print(f"{'Mean':<8} ${cv_mae.mean():>10,.0f} {cv_r2.mean():>8.4f}")
print(f"{'Std':<8} ${cv_mae.std():>10,.0f} {cv_r2.std():>8.4f}")
print(f"\n→ ${cv_mae.mean():,.0f} ± ${cv_mae.std():,.0f} MAE")
print(f"  {'✅ Stable!' if cv_mae.std() < 3000 else '⚠️ High variance across folds'}")

## Learning Curves

Diagnose bias vs variance: does the model need more data or more complexity?

In [ ]:
# ── Learning curves ───────────────────────────────────────────────────────
train_sizes, train_scores, val_scores = learning_curve(
    cv_pipe, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='neg_mean_absolute_error', n_jobs=-1
)

train_mae_lc = -train_scores.mean(axis=1) * 100_000
val_mae_lc = -val_scores.mean(axis=1) * 100_000
train_std = train_scores.std(axis=1) * 100_000
val_std = val_scores.std(axis=1) * 100_000

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(train_sizes, train_mae_lc - train_std, train_mae_lc + train_std,
                alpha=0.1, color='blue')
ax.fill_between(train_sizes, val_mae_lc - val_std, val_mae_lc + val_std,
                alpha=0.1, color='red')
ax.plot(train_sizes, train_mae_lc, 'b-o', label='Training MAE', markersize=5)
ax.plot(train_sizes, val_mae_lc, 'r-s', label='Validation MAE', markersize=5)
ax.axhline(y=40_000, color='green', linestyle='--', linewidth=1.5, label='$40k target')

ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('MAE ($)', fontsize=12)
ax.set_title('Learning Curve: Ridge (degree=2, α=1.0)\n'
             'Converging curves = good fit, gap = slight variance', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('img/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Prediction Intervals

Not just "$380k" but "$380k ± $45k with 95% confidence."

In [ ]:
# ── Bootstrap prediction intervals ────────────────────────────────────────
from sklearn.utils import resample

n_bootstrap = 100
boot_preds = np.zeros((n_bootstrap, len(X_test)))

for i in range(n_bootstrap):
    X_boot, y_boot = resample(X_train, y_train, random_state=i)
    boot_pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0))
    ])
    boot_pipe.fit(X_boot, y_boot)
    boot_preds[i] = boot_pipe.predict(X_test)

lower = np.percentile(boot_preds, 2.5, axis=0) * 100_000
upper = np.percentile(boot_preds, 97.5, axis=0) * 100_000
point_pred = y_pred * 100_000
actual = y_test.values * 100_000 if hasattr(y_test, 'values') else y_test * 100_000

coverage = np.mean((actual >= lower) & (actual <= upper)) * 100
avg_width = np.mean(upper - lower)
print(f"95% CI coverage: {coverage:.1f}% (target: 95%)")
print(f"Average interval width: ${avg_width:,.0f}")

In [ ]:
# ── Prediction intervals visualization ────────────────────────────────────
sort_idx = np.argsort(actual)[:100]

fig, ax = plt.subplots(figsize=(14, 6))
x_range = np.arange(len(sort_idx))

ax.fill_between(x_range, lower[sort_idx], upper[sort_idx],
                alpha=0.3, color='steelblue', label='95% CI')
ax.plot(x_range, point_pred[sort_idx], 'b-', linewidth=1, label='Predicted', alpha=0.7)
ax.scatter(x_range, actual[sort_idx], s=15, c='red', zorder=5, label='Actual', alpha=0.6)

ax.set_xlabel('District (sorted by value)', fontsize=12)
ax.set_ylabel('House Value ($)', fontsize=12)
ax.set_title('Prediction Intervals (95% Bootstrap CI)\n'
             'Red dots outside blue band = model uncertainty', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('img/prediction_intervals.png', dpi=150, bbox_inches='tight')
plt.show()

## Progress Summary

In [ ]:
# ── Final evaluation summary ──────────────────────────────────────────────
print("═" * 60)
print("  Ch.6 EVALUATION SUMMARY")
print("═" * 60)
print(f"\n  Model: Ridge (degree=2, α=1.0)")
print(f"  Test MAE:    ${mae:,.0f}")
print(f"  CV MAE:      ${cv_mae.mean():,.0f} ± ${cv_mae.std():,.0f}")
print(f"  R²:          {r2:.4f}")
print(f"  MAPE:        {mape:.1f}%")
print(f"  95% CI:      {coverage:.0f}% coverage, ±${avg_width/2:,.0f} avg half-width")
print(f"\n  Verdict: ✅ Model is validated and trustworthy")
print("═" * 60)
print("\n→ Next: Ch.7 optimizes hyperparameters systematically")


## Exercises

1. **Geographic bias**: Compute MAE separately for Northern CA (latitude > 37) vs Southern CA. Which region has higher errors?
2. **Luxury home analysis**: For homes >$400k (actual value), compute the mean residual. Is the model systematically over or underestimating?
3. **RMSE/MAE interpretation**: Our model has RMSE/MAE ≈ 1.37. What does this tell you about error distribution? Compare to a hypothetical model with RMSE/MAE = 1.05.
4. **Confidence interval width**: Bootstrap intervals are wider for expensive homes. Why? Compute average interval width for homes <$200k vs >$400k.


In [ ]:
# ── Exercise 1 scaffold: geographic bias ──────────────────────────────────
# TODO: Split test set by Latitude > 37, compute MAE for each region
# Hint: housing.feature_names includes 'Latitude'
pass


In [ ]:
# ── Exercise 2 scaffold: luxury home analysis ────────────────────────────
# TODO: Filter test_df for actual > 400_000, compute mean(residual)
# Negative mean = underestimate, positive = overestimate
pass


In [ ]:
# ── Exercise 3 scaffold: RMSE/MAE interpretation ─────────────────────────
# TODO: Our ratio is 1.37. Compute theoretical ratio for:
# - Uniform errors (all same magnitude)
# - One outlier scenario: [10k, 10k, 10k, ..., 100k]
pass


In [ ]:
# ── Exercise 4 scaffold: confidence interval width analysis ──────────────
# TODO: Group test_df by price segment, compute average interval width
# Use: width = upper - lower (from bootstrap intervals)
pass
